# FastVLA: Ultra-Efficient VLA Fine-Tuning on Tesla T4

FastVLA is designed to democratize Vision-Language-Action (VLA) models. In this notebook, we demonstrate how to fine-tune a 7.3 Billion parameter OpenVLA model on a free Tesla T4 (15GB) GPU using only ~6.3GB of VRAM.

### Why this matters?
- Standard OpenVLA-7B (FP16) requires ~28GB VRAM (impossible on T4).
- FastVLA uses 4-bit QLoRA and Custom Triton Kernels to reduce memory by 70%.
- You get 6x faster iterations than standard implementations.

> **Goal:** Fine-tune OpenVLA for 50 steps on the PushT robotics dataset in under 2 minutes.

## ⚠️ Important: Gated Model Access (Llama-2)

OpenVLA internally uses **Llama-2-7B** as its language backbone. Meta requires all users to manually request access to Llama-2. 

**If you haven't been approved yet:**
1. Visit [meta-llama/Llama-2-7b-hf](https://huggingface.co/meta-llama/Llama-2-7b-hf) and click **Request Access**.
2. **Wait for Approval**: This can take anywhere from 1 hour to 2 days.
3. **Login**: Once approved, use your HF token in the cell below.

**🚀 Want to skip the wait? (Instant Mode)**
If you want to start training **immediately** without waiting for Meta's approval, simply change the `model_id` in Step 2 to `"smolvla"`. It is non-gated and works instantly!

## 0. Authentication
1. **Add Token**: 
   - **Kaggle**: Go to **Add-ons -> Secrets**, add `HF_TOKEN` with your HuggingFace API key.
   - **Colab**: Click the **🔑 (Secrets)** icon, add `HF_TOKEN` and enable 'Notebook access'.

In [1]:
from huggingface_hub import login
import os

# Check for Kaggle Secrets or Colab Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    token = user_secrets.get_secret("HF_TOKEN")
    login(token=token)
except:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        login(token=token)
    except:
        # Manual fallback
        print("No HF_TOKEN found in secrets. Redirecting to manual login...")
        login()

## 1. Setup Environment
We install FastVLA and its optimized dependencies. 

**Tip for Kaggle users:** You can attach the 'openvla' model directly from the "+ Add Model" tab in the right sidebar to bypass the 15GB download and start training instantly!

In [2]:
# Install FastVLA directly from GitHub
!pip install -q git+https://github.com/BouajilaHamza/FastVLA.git

# Ensure optimized kernels and dependencies are present
!pip install -q triton bitsandbytes accelerate peft transformers datasets timm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Load Model in 4-bit (QLoRA)
We load OpenVLA-7B with 4-bit quantization. 

**Instant Start Tip:** To skip Llama-2 gated access issues, change `model_id` below to `"smolvla"`.

In [3]:
from fastvla import FastVLAModel
import torch

model_id = "openvla/openvla-7b" # Change to "smolvla" for non-gated, instant access

print(f"Loading {model_id} in 4-bit...")
model = FastVLAModel.from_pretrained(
    model_id,
    load_in_4bit=True,
    use_peft=True,
    gradient_checkpointing=True,
)

print(f"Model loaded. Current VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Loading openvla/openvla-7b in 4-bit...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded. Current VRAM: 0.71 GB


## 3. Fine-Tuning on PushT (Real Robotics)
Next, we load the lerobot/pusht_image dataset and run an optimized training loop. Watch the loss drop!

In [ ]:
from fastvla import FastVLATrainer, get_dataset

# Load only a small subset for demonstration
dataset = get_dataset("lerobot/pusht_image")

trainer = FastVLATrainer(
    model=model,
    dataset=dataset,
    batch_size=1,
    lr=1e-4,
    max_steps=50, # Set to 2000 for full convergence
)

print("Starting Training...")
results = trainer.train()

print(f"Training Complete! Average step latency: {results['avg_time_ms']:.2f} ms")

## 4. Inference (Predict Action)
Now we test the model by giving it an image and a text prompt to predict the robot's next action tensor.

In [ ]:
import numpy as np
from PIL import Image

# Dummy image for demonstration
dummy_image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
prompt = "push the t-shaped block into the goal zone"

action = model.predict(dummy_image, prompt)

print(f"Predicted Action Tensor (7D):")
print(action)

---
**Star the repo** to support democratized robotics!
[GitHub: FastVLA](https://github.com/BouajilaHamza/FastVLA)

**Created by the FastVLA Team.**